In [41]:
import requests
import pandas as pd
import numpy as np
import warnings
import os, json
from pathlib import Path
import datetime
from datetime import datetime as dt

warnings.filterwarnings("ignore")
 

DATA_DAILY = Path("data/daily")
OUTPUT_TOP = Path("output/top")

os.makedirs(DATA_DAILY, exist_ok=True)
os.makedirs(OUTPUT_TOP, exist_ok=True)

In [ ]:
# URLs
URL_GAINERS = "https://query1.finance.yahoo.com/v1/finance/screener/predefined/saved?scrIds=day_gainers&count=250"
URL_ACTIVE = "https://query1.finance.yahoo.com/v1/finance/screener/predefined/saved?scrIds=most_actives&count=250"
URL_TRENDING = "https://query1.finance.yahoo.com/v1/finance/screener/predefined/saved?scrIds=trending_tickers&count=250"

def fetch_screener(scrId, count=250): 
    url = f"https://query1.finance.yahoo.com/v1/finance/screener/predefined/saved?scrIds={scrId}&count={count}"
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
        data = r.json()["finance"]["result"][0]["quotes"]
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error loading screener {scrId}: {e}")

def fetch_trending():
    url = "https://query1.finance.yahoo.com/v1/finance/trending/US"
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
        data = r.json()["finance"]["result"][0]["quotes"]
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error loading trending {e}")

In [43]:
import re

def filter_equities(df):
    eq = []
    for s in df["symbol"]:
        if re.fullmatch(r"[A-Za-z][A-Za-z0-9\.-]{0,9}", s):
            eq.append(s)
    return pd.DataFrame({"symbol": eq})

print("Downloading lists...")

df_gainers = fetch_screener("day_gainers")
df_active  = fetch_screener("most_actives")
df_trending_raw = fetch_trending()
df_trending = filter_equities(df_trending_raw)


print("Gainers:", len(df_gainers))
print("Most Active:", len(df_active))
print("Trending:", len(df_trending))

combined = pd.concat([df_gainers, df_active, df_trending], ignore_index=True)
combined.drop_duplicates(subset="symbol", inplace=True)

print("Total unique hot symbols:", len(combined))


Gainers: 191
Most Active: 243
Trending: 20
Total unique hot symbols: 415


In [44]:
def fix_num(x):
    """
    Convert formatted numbers:
    '1.2M', '330K', '2.3B' → float
    """
    if x is None:
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)

    x = str(x).replace(",", "")

    try:
        if x.endswith("M"):
            return float(x[:-1]) * 1e6
        if x.endswith("B"):
            return float(x[:-1]) * 1e9
        if x.endswith("K"):
            return float(x[:-1]) * 1e3
        return float(x)
    except:
        return np.nan

numeric_cols = [
    "regularMarketPrice",
    "regularMarketChangePercent",
    "regularMarketChange",
    "regularMarketVolume",
    "averageDailyVolume3Month",
    "marketCap"
]

for col in numeric_cols:
    if col in combined.columns:
        combined[col] = combined[col].apply(fix_num)
    
df = combined.copy()

# Volume Spike Score
df["VolumeSpike"] = df["regularMarketVolume"] / df["averageDailyVolume3Month"]
df["VolumeScore"] = df["VolumeSpike"].rank(pct=True)

# Momentum Score (based on daily % change)
df["MomentumScore"] = df["regularMarketChangePercent"].rank(pct=True)

# Volatility Score (higher movement = more hot)
df["VolatilityScore"] = df["regularMarketChange"].abs().rank(pct=True)

# Trend Score (proxy: price vs avg volume)
df["TrendScore"] = (df["regularMarketPrice"] / (df["averageDailyVolume3Month"] + 1)).rank(pct=True)

# Final score
df["HotScore"] = (
    0.35 * df["MomentumScore"] +
    0.35 * df["VolumeScore"] +
    0.20 * df["VolatilityScore"] +
    0.10 * df["TrendScore"]
)

df.sort_values("HotScore", ascending=False, inplace=True)

# Keep relevant columns
output_df = df[[
    "symbol",
    "regularMarketPrice",
    "regularMarketChangePercent",
    "regularMarketVolume",
    "averageDailyVolume3Month",
    "marketCap",
    "VolumeSpike",
    "MomentumScore",
    "VolumeScore",
    "VolatilityScore",
    "TrendScore",
    "HotScore"
]]

top40 = output_df.head(40)
top40.head(10)

,symbol,regularMarketPrice,regularMarketChangePercent,regularMarketVolume,averageDailyVolume3Month,marketCap,VolumeSpike,MomentumScore,VolumeScore,VolatilityScore,TrendScore,HotScore
0,DPC,45.785,38.742424,14979709.0,300.0,6.454322e+09,49932.363333,1.000000,1.000000,0.911548,1.000000,0.982310
3,AYI,363.180,18.876625,1042079.0,399325.0,1.100986e+10,2.609601,0.992629,0.970516,0.990172,0.965602,0.981695
14,ASND,260.470,9.436582,2000720.0,671601.0,1.708061e+10,2.979031,0.965602,0.975430,0.963145,0.928747,0.964865
7,KYMR,113.560,13.707815,3189384.0,635833.0,9.341171e+09,5.016072,0.982801,0.995086,0.921376,0.874693,0.964005
2,TECH,70.555,19.838642,45507067.0,2732245.0,1.104671e+10,16.655559,0.995086,0.997543,0.896806,0.680590,0.944840
11,AMAT,650.620,10.468359,9630525.0,7787708.0,5.165659e+11,1.236631,0.972973,0.872236,0.992629,0.823096,0.926658
6,MU,1214.400,15.886706,66495603.0,51095566.0,1.369520e+12,1.301397,0.985258,0.889435,0.997543,0.665848,0.922236
41,MKSI,402.900,5.576228,1723317.0,1182782.0,2.721353e+10,1.457003,0.904177,0.916462,0.958231,0.921376,0.921007
28,BIO,298.810,6.698808,458990.0,378648.0,7.996633e+09,1.212181,0.933661,0.864865,0.953317,0.960688,0.916216
17,GLW,222.805,8.247092,19733316.0,13141064.0,1.917544e+11,1.501653,0.955774,0.926290,0.941032,0.628993,0.909828


In [45]:
import plotly.express as px


fig = px.scatter(
    top40.sort_values(by=["regularMarketPrice", "HotScore"] ,ascending=True),
    x="TrendScore",
    y="HotScore",
    size="regularMarketPrice",
    color="HotScore",
    text="regularMarketPrice",
    hover_name="symbol",
    color_continuous_scale="ice",
    size_max=40
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(width=1, color="white"))
)

fig.update_layout(
    title="Top 40 Swing Stocks (High-Confidence Set)",
    xaxis_title="Trend Score",
    yaxis_title="Hot Score",
    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",

    height=1000,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()
 
chart_path = os.path.join(OUTPUT_TOP, "scatter_trend_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [46]:
fig = px.scatter(
    top40.sort_values(by=["regularMarketPrice", "HotScore"] ,ascending=True),
    x="VolatilityScore",
    y="HotScore",
    size="TrendScore",
    color="MomentumScore",
    text="regularMarketPrice",
    hover_name="symbol",
    color_continuous_scale="ice",
    size_max=40
)

fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(width=1, color="white"))
)

fig.update_layout(
    title="Top 40 Stocks (High-Confidence Set)",
    xaxis_title="Volatility Score",
    yaxis_title="Hot Score",
    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",

    height=1000,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show ()

chart_paths = os.path.join(OUTPUT_TOP, "scatter_volatility.html")
fig.write_html(chart_paths, include_plotlyjs="cdn")

In [47]:
fig = px.bar(
    top40.sort_values(by=["regularMarketPrice", "HotScore"] ,ascending=True),
    x="HotScore",
    y="symbol",
    orientation="h",
    color="TrendScore",
    text="regularMarketPrice",
    hover_name="symbol",
    color_continuous_scale="Blues",
    hover_data=["marketCap"],
)

#fig.update_traces(texttemplate="%{x:.3f}", textposition="outside")
fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 40 Swing Stocks by Hot Score",
    xaxis_title="Hot Score",
    yaxis_title="Symbol",
    plot_bgcolor="#010136",
    paper_bgcolor="#101029",
    font_color="white",
    height=1200,
    margin=dict(l=50, r=50, t=50, b=50),
)

fig.show()

chart_path = os.path.join(OUTPUT_TOP, "bar_hot_score.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [48]:
fig = px.bar(    
    top40.sort_values(by=["regularMarketPrice", "TrendScore"], ascending=True),
    x="TrendScore",
    y="symbol",
    orientation="h",
    color="HotScore",
    color_continuous_scale="ice",
    hover_data=[
        "regularMarketPrice",
        "marketCap",
    ],
    text="regularMarketPrice"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="inside")

fig.update_layout(
    title="Top 40 Stocks (Ranked by Trend Score)",
    xaxis_title="Trend Score",
    yaxis_title="Symbol",
    plot_bgcolor="#010136",
    paper_bgcolor="#101029",
    font_color="white",
    height=1200,
    margin=dict(l=50, r=50, t=50, b=50)
) 

fig.show()
 
chart_paths = os.path.join(OUTPUT_TOP, "bar_trend_score.html")
fig.write_html(chart_paths, include_plotlyjs="cdn")

In [49]:
fig = px.treemap(
    top40.sort_values(by=["regularMarketPrice", "TrendScore"], ascending=True),
    path=["symbol", "regularMarketPrice"],
    values="HotScore",
    color="HotScore",
    color_continuous_scale="ice",
    hover_data=[
        "regularMarketPrice",
        "marketCap",
    ],
)


fig.update_layout(
    title="Market Cap Distribution by Stock",
    paper_bgcolor="#0b0f1a",
    plot_bgcolor="#0b0f1a",
    font_color="white",
    height=800,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.show()

chart_paths = os.path.join(OUTPUT_TOP, "treemap_market_cap.html")
fig.write_html(chart_paths, include_plotlyjs="cdn")

In [50]:

try:
    top40  # DataFrame with columns including symbol, HotScore, regularMarketPrice, regularMarketChangePercent, regularMarketVolume, VolumeSpike
except NameError:
    raise RuntimeError("Expected DataFrame 'top40' not found. Rename your top40 var or adapt this block.")

timestamp = dt.now().strftime("%Y%m%d%H%M%S")

top40_file = DATA_DAILY / f"hot_stocks_{timestamp}.csv"
top40.to_csv(top40_file, index=False)